<a href="https://colab.research.google.com/github/sandy0708/Song-Lyrics-Generator/blob/main/K_POP_%EB%85%B8%EB%9E%98%EA%B0%80%EC%82%AC_%EC%83%9D%EC%84%B1%EA%B8%B0_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎤 🎧 💿 K-POP 세대별 노래 가사 생성기 💿 🎧 🎤

###  🪄 준비 과정! 코드를 모두 돌려주세요 
  
⬇️ 괄호 사이의 재생 버튼을 눌러주세요 !! *✨ 한 번 실행하시면 이 부분은 다시 실행하실 필요 없습니다!!! ✨*

In [ ]:
!pip install transformers
import os
# from google.colab import drive

import pandas as pd
import numpy as np
import random
import re

import torch
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler, SequentialSampler
torch.manual_seed(42)

from transformers import GPT2LMHeadModel,  GPT2Tokenizer, GPT2Config, GPT2LMHeadModel
from transformers import AdamW, get_linear_schedule_with_warmup

from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
import nltk
nltk.download('punkt')

In [ ]:
import time 
!if [ -e "/content/model.zip" ]; then rm "/content/model.zip"; fi
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1YmEr0ZAMo59LGXZitLiiWoPJA6iQ6g2q' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1YmEr0ZAMo59LGXZitLiiWoPJA6iQ6g2q" -O model.zip && rm -rf /tmp/cookies.txt
time.sleep(2)
!if [ -d "/content/model_save" ]; then rm -r "/content/model_save"; fi
!unzip /content/model.zip
time.sleep(1)
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1C2D9jxNT00_vp_t_mSegMxFkGkARLx01' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1C2D9jxNT00_vp_t_mSegMxFkGkARLx01" -O tfidf.py && rm -rf /tmp/cookies.txt

In [ ]:
#GPT2 모델, 토크나이저 불러오기

LOAD_MODEL_DIR = "/content/model_save"
model = GPT2LMHeadModel.from_pretrained(LOAD_MODEL_DIR)
from transformers import BertTokenizerFast, GPT2LMHeadModel
tokenizer = BertTokenizerFast.from_pretrained(LOAD_MODEL_DIR)

In [ ]:
from tfidf import Tfidf
make_title = Tfidf().make_title # 제목 붙이기 

### 본격적인 가사 생성! 🎶 🎵

In [ ]:

#@markdown ⬅️ 아래 항목들을 채우고 click!

#@markdown 원하는 가사의 첫 문장과 세대를 입력해주세요:

generation = "3\uC138\uB300" #@param ['1세대', '2세대', '3세대']
lyrics = "\uBA38\uB9AC\uBD80\uD130 \uBC1C\uB05D\uAE4C\uC9C0 " #@param {type:"string"}

#@markdown 생성할 가사의 개수를 고르세요:
num = 3 #@param {type:"slider", min:1, max:10, step:1}

#input 생성하기
given = '<|startoftext|>' + '<' + generation + '> ' + lyrics + '<br>'
input = tokenizer.encode(given, return_tensors='pt')
input = input[:, 1:]  # remove cls token

#output 만들기
sample_outputs = model.generate(
                                input, 
                                #bos_token_id=random.randint(1,30000),
                                do_sample=True,   
                                top_k=50, 
                                top_p=0.95, 
                                max_length = 300,
                                num_return_sequences=num,
                                repetition_penalty=1.3
                                )

# 생성된 가사 print
for i, sample_output in enumerate(sample_outputs):
    lyric = tokenizer.decode(sample_output, skip_special_tokens=True)
    lyric2 = lyric.replace(' <br> ','\n')
    lyric2 = lyric2.replace('<br>','')
    print("\n" + ("-"*50) + "[" + str(i+1) + "]" + ("-"*50) + "\n")
    print("🎧 세대: ", generation,'\n' "🎤 입력된 가사: ", lyrics,'\n')
    print('💿 제목: ', make_title(lyric.replace(' <br> ',' ')),'\n')
    print(lyric2)